# Video Compression vs YOLOv8n Object Detection

End-to-end experiment notebook:

1. **Compress** each input video with H.264 (libx264) at CRF 18 / 28 / 38
2. **Detect** with YOLOv8n on the original + 3 compressed versions
3. **Record** file size, total detections, and average confidence per version
4. **Plot** the three headline charts for the paper

The notebook wires together the functions in `scripts/` so the logic stays in one place.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.compress import (
    CRF_LEVELS,
    COMPRESSED_DIR,
    compress_all,
    discover_input_videos,
)
from scripts.detect import run_detection  # noqa: F401 (available for ad-hoc calls)
from scripts.run_experiment import (
    RESULTS_CSV,
    CSV_COLUMNS,  # noqa: F401
    evaluate_version,
    write_csv,
)

print('CRF levels:', CRF_LEVELS)
print('Results CSV will be written to:', RESULTS_CSV.relative_to(REPO_ROOT))

## 2. Discover input videos

Drop your short clips in `data/input_videos/` (a sample `d.mp4` is already there).

In [ ]:
videos = discover_input_videos()
for v in videos:
    print(f'  {v.relative_to(REPO_ROOT)}  ({v.stat().st_size / 1024:.1f} KB)')
assert videos, 'No videos found in data/input_videos/ — add some .mp4 files first.'

## 3. Compress each video with libx264

Produces `data/compressed/<stem>/compressed_{high,medium,low}.mp4`. If the files already exist we reuse them — delete that folder to force a re-encode.

In [ ]:
compressed_paths: dict[str, dict[str, Path]] = {}

for video in videos:
    out_dir = COMPRESSED_DIR / video.stem
    expected = {lvl: out_dir / f'compressed_{lvl}.mp4' for lvl in CRF_LEVELS}
    if all(p.exists() for p in expected.values()):
        print(f'[skip] {video.name} — already compressed')
        compressed_paths[video.stem] = expected
    else:
        compressed_paths[video.stem] = compress_all(video)

## 4. Run YOLOv8n on every (video, version) and build the results table

In [ ]:
rows: list[dict] = []

for video in videos:
    print(f'\n=== {video.name} ===')
    rows.append(evaluate_version(
        source_video=video, version='original', codec='original', crf='', video_path=video,
    ))
    for level, crf in CRF_LEVELS.items():
        rows.append(evaluate_version(
            source_video=video,
            version=level,
            codec='h264',
            crf=crf,
            video_path=compressed_paths[video.stem][level],
        ))

csv_path = write_csv(rows)
print(f'\nWrote {len(rows)} rows -> {csv_path.relative_to(REPO_ROOT)}')

## 5. Inspect the results table

In [ ]:
from IPython.display import display
import pandas as pd

df = pd.read_csv(csv_path)
display(df[['source_video', 'version', 'codec', 'crf',
            'file_size_mb', 'total_detections', 'avg_confidence']])

## 6. Plots

Three simple line charts, one per metric, also saved to `outputs/results/plots/`.

In [ ]:
from scripts.plot_results import main as plot_main
plot_main()

In [ ]:
from IPython.display import Image, display
for name in ('file_size.png', 'total_detections.png', 'avg_confidence.png'):
    display(Image(filename=str(REPO_ROOT / 'outputs' / 'results' / 'plots' / name)))

## 7. (Optional) Show the annotated sample frames

One frame per version, grabbed during detection.

In [ ]:
from IPython.display import Image, display, Markdown

for video in videos:
    display(Markdown(f'### {video.name}'))
    for version in ['original', 'high', 'medium', 'low']:
        sample = REPO_ROOT / 'outputs' / 'annotated' / video.stem / f'{version}_sample.jpg'
        if sample.exists():
            display(Markdown(f'**{version}**'))
            display(Image(filename=str(sample), width=480))